# Module 6: Multi-Agent (Optional) (15 min)

> **Optional module.** You've already built and deployed a complete agent in Modules 1–5. This module adds delegation on top of that same agent.

Add agent delegation — when the customer service agent encounters a technical issue, it escalates to a **tech support specialist** agent. This is the agents-as-tools pattern: one orchestrator calls specialists like functions.

**Prerequisites:** Modules 1-4 completed

In [1]:
!pip install -q -r requirements.txt

In [ ]:
# AWS-sponsored events / AWS credits
# If you are running this workshop with AWS-provided credits, those credits
# only work with Amazon Nova models — not Claude (the default).
# To switch, import BedrockModel and pass it to Agent(...):
#
# from strands.models import BedrockModel
# model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# agent = Agent(model=model, tools=[...], system_prompt=...)
#
# Available Nova model IDs: https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-amazon.html
#   amazon.nova-micro-v1:0  — fastest, text-only, lowest cost
#   amazon.nova-lite-v1:0   — low-cost, multimodal (text, image, video)
#   amazon.nova-pro-v1:0    — balanced accuracy/speed, multimodal (recommended)
#
# Run locally without AWS credentials using Ollama (https://ollama.com/download):
#   1. Install Ollama — macOS: DMG at ollama.com | Linux: curl -fsSL https://ollama.com/install.sh | sh
#   2. ollama pull llama3.1   (supports tool use)
#   3. pip install strands-agents[ollama]
#   4. from strands.models import OllamaModel
#      model = OllamaModel(host="http://localhost:11434", model_id="llama3.1")
#      agent = Agent(model=model, tools=[...], system_prompt=...)
#   Other models with tool support: llama3.2, qwen2.5, qwen3, mistral

---

## Part 1: Build the Tech Support Specialist

A focused agent with its own tools and system prompt. It handles device troubleshooting, connectivity issues, and firmware updates.

In [2]:
from strands import Agent, tool
from customer_service_tools import lookup_customer, get_order_history, process_refund


# --- Tech support tools ---

@tool
def check_device_compatibility(device: str, issue: str) -> str:
    """Check if a device has known compatibility issues.

    Args:
        device: The device name or model
        issue: Description of the issue
    """
    known_issues = {
        "Wireless Headphones": "Known Bluetooth 5.0 pairing issue with older devices. Fix: Reset headphones (hold power 10s), then re-pair.",
        "USB-C Hub": "Some laptops require USB-C alt mode. Check laptop specs for DisplayPort over USB-C support.",
        "Mechanical Keyboard": "Firmware v2.1 has a key ghosting bug. Update to v2.3 via manufacturer website.",
    }
    for device_name, fix in known_issues.items():
        if device_name.lower() in device.lower():
            return f"Known issue found for {device_name}: {fix}"
    return f"No known issues found for '{device}'. Recommend standard troubleshooting: restart device, check connections, update drivers."


@tool
def run_diagnostic(device: str) -> str:
    """Run a remote diagnostic check on a device.

    Args:
        device: The device name or model to diagnose
    """
    return (
        f"Diagnostic results for {device}:\n"
        f"- Firmware: v2.1 (update available: v2.3)\n"
        f"- Connection: Stable\n"
        f"- Battery: 85%\n"
        f"- Last sync: 2 hours ago\n"
        f"Recommendation: Update firmware to resolve known issues."
    )


print("✅ Tech support tools defined")

✅ Tech support tools defined


---

## Part 2: Wrap the Specialist as a Tool

The `@tool` decorator turns the specialist agent into a callable tool for the orchestrator. The orchestrator decides when to delegate.

In [3]:
@tool
def tech_support_specialist(issue_description: str) -> str:
    """Escalate a technical issue to the tech support specialist agent.
    Use this when a customer has a device problem, connectivity issue,
    or needs technical troubleshooting beyond basic order/account help.

    Args:
        issue_description: Detailed description of the technical issue including device name and symptoms
    """
    specialist = Agent(
        tools=[check_device_compatibility, run_diagnostic],
        system_prompt="""You are a tech support specialist for an electronics store.
You diagnose device issues, check compatibility, and provide step-by-step fixes.
Be technical but clear. Always provide actionable next steps.""",
        callback_handler=None,  # Silent — don't stream to user
    )

    print(f"\n[DELEGATION] 🔧 Tech support specialist activated")
    print(f"[DELEGATION] 📋 Issue: {issue_description[:80]}...")

    response = specialist(issue_description)

    print(f"[DELEGATION] ✅ Specialist responded")
    return str(response)


print("✅ tech_support_specialist tool defined")

✅ tech_support_specialist tool defined


---

## Part 3: The Orchestrator Agent

The customer service agent now has `tech_support_specialist` as one of its tools. It decides when to escalate.

In [4]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

You handle:
- Account lookups and order status
- Refund processing
- Basic questions

For TECHNICAL issues (device problems, connectivity, firmware, troubleshooting),
delegate to the tech_support_specialist tool. Provide it with the device name
and a clear description of the issue.

After getting the specialist's response, relay the solution to the customer
in a friendly, non-technical way."""

orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
)

# This should trigger delegation to the tech support specialist
result = orchestrator(
    "I'm customer C-1001. The wireless headphones I bought aren't pairing "
    "with my phone. I've tried restarting them but nothing works."
)

I'll look up your account and connect you with our tech support specialist at the same time to get this sorted out quickly!
Tool #1: lookup_customer

Tool #2: tech_support_specialist

[DELEGATION] 🔧 Tech support specialist activated
[DELEGATION] 📋 Issue: Customer has wireless headphones that won't pair with their phone. They have alr...
[DELEGATION] ✅ Specialist responded
Hi Sarah! I've pulled up your account and looped in our tech support specialist. 😊

To help get your headphones pairing properly, our specialist needs a couple more details:

1. **What model are your wireless headphones?**
2. **What phone model are you using?**

Once we have that info, we can dig into any known issues and get you connected!

In [5]:
# This should NOT trigger delegation — it's a simple order question
orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
)

result = orchestrator("I'm customer C-1002. Where is my keyboard order?")

Let me pull up your account and order history at the same time!
Tool #1: lookup_customer

Tool #2: get_order_history
Hi Mike! Here's the update on your keyboard order:

- **Order:** ORD-5390 — Mechanical Keyboard ($149.99)
- **Status:** ⚠️ **Delayed**
- **Originally Estimated Delivery:** April 25, 2025
- **Tracking Number:** TRK-776655

Unfortunately, it looks like your order has been delayed. I'm sorry about that! You can use tracking number **TRK-776655** to get the latest shipping updates.

Is there anything else I can help you with — such as a refund or further assistance with this order?

---

## 🎯 Try It Yourself

Try different scenarios to see when the orchestrator delegates vs handles directly:

In [6]:
# Try these:
# "My USB-C hub isn't showing my external monitor" → should delegate
# "I want a refund for order ORD-5521" → should handle directly
# "The keyboard keys are sticking and some don't register" → should delegate

orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
)

result = orchestrator("I'm C-1002. The mechanical keyboard keys are ghosting — I press one key and two characters appear.")

I'll look up your account and get the tech support specialist working on your keyboard issue at the same time!
Tool #1: lookup_customer

Tool #2: tech_support_specialist

[DELEGATION] 🔧 Tech support specialist activated
[DELEGATION] 📋 Issue: Customer has a mechanical keyboard experiencing key ghosting — when one key is p...
[DELEGATION] ✅ Specialist responded
Hi Mike! I've pulled up your account and got an answer from our tech specialist. Here's what's going on and how to fix it:

---

**The Good News:** Your keyboard's ghosting issue is caused by a known bug in an **outdated firmware version (v2.1)**. A fix is already available — just update to **firmware v2.3**!

### Here's what to do:

1. **Update the Firmware (Main Fix)**
   - Go to your keyboard manufacturer's official website → Support → Downloads → Firmware
   - Download **v2.3** for your keyboard model
   - Connect the keyboard via **USB cable** (not wireless) to run the update
   - Follow the on-screen steps and **don't unplug

---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the orchestrator — which delegates technical issues to the specialist on any turn — run the companion script in a **terminal**. From the cloned repo:

```bash
cd samples/06-multi-agent
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. The orchestrator keeps its context across turns and routes each message to the specialist or handles it directly.

---

## What's Next

The agent can now delegate to specialists. But how do you know it's working correctly at scale? In **Module 7: Evals**, you'll write automated evaluations to test the agent's behavior.